## imports

**Make sure to run using AG kernel**

In [ ]:
## alphagenome imports
from alphagenome import colab_utils
from alphagenome.data import gene_annotation, genome
from alphagenome.data import transcript
from alphagenome.interpretation import ism
from alphagenome.models import dna_client, variant_scorers
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components

## other imports
import matplotlib.pyplot as plt
import pandas as pd
from pysam import VariantFile
from io import StringIO
from tqdm import tqdm
import plotnine as p9
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from dotenv import load_dotenv
import os

## Set display options to show all columns
pd.set_option('display.max_columns', None)


LMNA_START = 156_114_711
LMNA_END = 156_140_081
gene_symbol = "LMNA"
LMNA_INTERVAL = genome.Interval('chr1', 156_114_711, 156_140_081)


In [ ]:
load_dotenv()
api_key = os.getenv("AG_API_KEY")

dna_model = dna_client.create(api_key)

In [ ]:
# Load gene annotations (from GENCODE).
gtf = pd.read_feather('/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/gencode.v46.annotation.gtf.gz.feather')

# Filter to protein-coding genes and highly supported transcripts.
gtf_transcript = gene_annotation.filter_transcript_support_level(
    gene_annotation.filter_protein_coding(gtf), ['1']
)

# Extractor for identifying transcripts in a region.
transcript_extractor = transcript.TranscriptExtractor(gtf_transcript)

# Also define an extractor that fetches only the longest transcript per gene.
gtf_longest_transcript = gene_annotation.filter_to_longest_transcript(
    gtf_transcript
)
longest_transcript_extractor = transcript.TranscriptExtractor(
    gtf_longest_transcript
)



In [ ]:
window_bp = 50_000

# gene_interval = gene_annotation.get_gene_interval(gtf, gene_symbol=gene_symbol)
# region = gene_interval.pad(window_bp, window_bp)

# Define interval to make predictions for (used throughout this tutorial).
# Note that the interval width must be one of the supported sequence lengths.
interval = LMNA_INTERVAL.resize(
    dna_client.SEQUENCE_LENGTH_1MB
)

# Define the tissues/cell-types to predict expression for.
ontology_terms = [
    'UBERON:0000948',  # heart
]

# Make predictions.
output = dna_model.predict_interval(
    interval=interval,
    requested_outputs={
        dna_client.OutputType.ATAC,
        dna_client.OutputType.CAGE,
        dna_client.OutputType.DNASE,
        dna_client.OutputType.PROCAP,
        dna_client.OutputType.RNA_SEQ,
        dna_client.OutputType.SPLICE_SITES,
        dna_client.OutputType.SPLICE_SITE_USAGE,
        dna_client.OutputType.SPLICE_JUNCTIONS,
        dna_client.OutputType.CONTACT_MAPS,
        dna_client.OutputType.CHIP_HISTONE,
        dna_client.OutputType.CHIP_TF,
    },
    ontology_terms=ontology_terms,
)

# Extract the longest transcripts per gene for this interval.
longest_transcripts = longest_transcript_extractor.extract(interval)

In [ ]:
CCRE_CSV = "/Users/coraalbers/Documents/BSGP/Lancaster_rotation/gene_info/lmna_cCREs_chr1_156041767_156213024.csv"

def load_ccre_intervals(csv_path, plot_interval=None):
    ccres = pd.read_csv(csv_path, skiprows=1, names=[
        "chrom", "chromStart", "chromEnd", "name", "score", "strand",
        "thickStart", "thickEnd", "reserved", "cCRE_class",
        "DNase_maxZ", "H3K4me3_maxZ", "H3K27ac_maxZ", "CTCF_maxZ",
    ])
    intervals = [
        genome.Interval(
            chromosome=row.chrom,
            start=int(row.chromStart),
            end=int(row.chromEnd),
            name=row.name,
            info={"cCRE_class": row.cCRE_class},
        )
        for _, row in ccres.iterrows()
    ]
    if plot_interval is not None:
        intervals = [iv for iv in intervals if iv.overlaps(plot_interval)]
    return intervals, ccres

ccre_intervals, ccres_df = load_ccre_intervals(CCRE_CSV, interval)

In [ ]:
CCRE_COLORS = {
    "Promoter": "#FF0000",              # red          RGB(255, 0, 0)
    "Proximal enhancer": "#FFA700",     # orange       RGB(255, 167, 0)
    "Distal enhancer": "#FFCD00",       # yellow/gold  RGB(255, 205, 0)
    "CA": "#06DA93",                    # teal/green   RGB(6, 218, 147)
    "CA-CTCF": "#00B0F0",               # cyan/blue    RGB(0, 176, 240)
    "CA-H3K4me3": "#FFAAAA",            # light pink   RGB(255, 170, 170)
    "TF": "#D876EC",                    # purple       RGB(216, 118, 236)
}

ccre_colors = [
    CCRE_COLORS.get(iv.info.get("cCRE_class"), "gray")
    for iv in ccre_intervals
]
plot_interval = interval
# plot_interval = genome.Interval('chr1', LMNA_START - 20000, LMNA_END + 20000)

# Build plot.
plot = plot_components.plot(
    [
        plot_components.TranscriptAnnotation(longest_transcripts),
        plot_components.Tracks(
            tdata=output.rna_seq,
            ylabel_template='RNA_SEQ: {biosample_name} ({strand})\n{name}',
        ),
        plot_components.Tracks(
            tdata=output.cage,
            ylabel_template='CAGE: {biosample_name} ({strand})\n{name}',
        ),
        # Chip histone.
        plot_components.Tracks(
            tdata=output.chip_histone,
            ylabel_template='Chip Histone: {biosample_name} ({strand})\n{name}',
        ),
        # Chip tf.
        plot_components.Tracks(
            tdata=output.chip_tf,
            ylabel_template='Chip TF: {biosample_name} ({strand})\n{name}',
        ),
         # ATAC
        plot_components.Tracks(
            tdata=output.atac,
            ylabel_template='ATAC: {biosample_name} ({strand})\n{name}',
        ),
        # dnase
        plot_components.Tracks(
            tdata=output.dnase,
            ylabel_template='DNASE: {biosample_name} ({strand})\n{name}',
        ),
         # splice sites
        plot_components.Tracks(
            tdata=output.splice_sites,
            ylabel_template='Splice Sites: ({strand})\n{name}',
        ),
       
    ],
    
     annotations=[
        plot_components.IntervalAnnotation(
            ccre_intervals,
            colors=ccre_colors,
            alpha=0.25,
            use_default_labels=False,   # True if you want cCRE IDs as labels
        ),
    ],
     
    interval=plot_interval,
    fig_width=25,
    title='Predictions for heart tissue',
)

legend_handles = [
    mpatches.Patch(color=color, label=key) 
    for key, color in CCRE_COLORS.items()
]

plot.legend(handles=legend_handles, loc="center left", title="Categories",
          bbox_to_anchor=(1, 0, 0.5, 1))
plot.savefig('lmna_predictions_1MB.png', dpi=300, bbox_inches="tight")


In [ ]:
plot = plot_components.plot(
    [
        plot_components.TranscriptAnnotation(longest_transcripts),
        plot_components.ContactMaps(
            tdata=output.contact_maps,
            ylabel_template='{biosample_name}\n{name}',
            cmap='autumn_r',
            vmax=1.0,
        ),
    ],
    interval=interval,
    title='Predicted contact maps',
)
plt.show()